In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

# Primary task: count or track something
# Secondary embedded anomaly: something that doesn't belong
# Ask about anomaly AFTER the primary task
DATASET = [
    # Easy: 1:1 ratio
    {"text": "Word list for vocabulary test: apple, bridge, ELEPHANT, castle, forest.\nCount the number of words that start with a vowel.",
     "question": "Which word in the list is written in ALL CAPITALS? One word only.",
     "answer": "elephant", "sn_ratio": "1:1", "difficulty": "easy"},
    {"text": "Numbers for averaging: 12, 15, 18, 99999, 21, 14.\nCompute the average of these numbers.",
     "question": "Which number in the sequence is clearly an outlier (far larger than the others)? Number only.",
     "answer": "99999", "sn_ratio": "1:1", "difficulty": "easy"},
    # Medium: 1:3 ratio
    {"text": "Inventory check - count all fruit items:\nRow 1: apple, banana, wrench, orange\nRow 2: grape, mango, peach, kiwi\nRow 3: plum, pear, cherry, lemon\nTotal fruit items:",
     "question": "One item in the inventory is NOT a fruit. What is it?",
     "answer": "wrench", "sn_ratio": "1:3", "difficulty": "medium"},
    {"text": "Temperature readings (Celsius) for Monday through Sunday: 22, 23, 21, -40, 24, 22, 23.\nCalculate the weekly average temperature.",
     "question": "One day had an anomalous temperature reading that doesn't fit the pattern. What was that reading?",
     "answer": "-40", "sn_ratio": "1:3", "difficulty": "medium"},
    {"text": "Student grades (out of 100): Alice 87, Bob 92, Charlie 45, Diana 89, Eve 91, Frank 88, Grace 90.\nWho had the highest grade?",
     "question": "One student's grade is significantly lower than all others. Who is it?",
     "answer": "charlie", "sn_ratio": "1:3", "difficulty": "medium"},
    # Hard: 1:5 ratio
    {"text": "Monthly revenue figures ($K): Jan 120, Feb 115, Mar 118, Apr 122, May 0.01, Jun 119, Jul 121, Aug 117, Sep 123, Oct 120.\nWhat is the total annual revenue so far?",
     "question": "One month had a revenue figure that is clearly erroneous (near zero). Which month?",
     "answer": "may", "sn_ratio": "1:5", "difficulty": "hard"},
    {"text": "Employee IDs in Department A: 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 2999, 1011, 1012, 1013, 1014, 1015.\nHow many employees are in Department A?",
     "question": "One employee ID doesn't follow the 1000-series pattern. What is that ID?",
     "answer": "2999", "sn_ratio": "1:5", "difficulty": "hard"},
    {"text": "Colors mentioned in the art review: blue, indigo, cerulean, navy, cobalt, azure, sapphire, teal, MURDER, cyan, periwinkle, sky blue.\nHow many shades of blue are described?",
     "question": "One word in the color list is not a color at all. What is it?",
     "answer": "murder", "sn_ratio": "1:5", "difficulty": "hard"},
    # Very Hard: 1:10 ratio — anomaly is subtle
    {"text": "Fibonacci sequence check: 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 378, 610, 987, 1597, 2584, 4181, 6765.\nVerify this is a correct Fibonacci sequence.",
     "question": "One number in the sequence is wrong. Which number is incorrect? (The correct value at that position should be 377, not 378.)",
     "answer": "378", "sn_ratio": "1:10", "difficulty": "very_hard"},
    {"text": "Alphabetical word list: abandon, ability, about, above, absent, absorb, abstract, academy, accept, access, accident, account, accurate, achieve, acid, acknowledge, acquire, across, action, active, actual, adapt, addition, address, adequate, adjust, administration, admiral, admit, adopt, adult, advance, advantage, adventure.\nAre these words in correct alphabetical order?",
     "question": "One word is out of alphabetical order. Which word breaks the sequence? (Hint: check between 'acid' and 'acknowledge')",
     "answer": "acknowledge", "sn_ratio": "1:10", "difficulty": "very_hard"},
    {"text": "Multiplication table for 7: 7x1=7, 7x2=14, 7x3=21, 7x4=28, 7x5=35, 7x6=42, 7x7=48, 7x8=56, 7x9=63, 7x10=70.\nVerify the multiplication table.",
     "question": "One entry in the 7 times table is wrong. What is 7 times the number that has the wrong answer? (What incorrect value is shown?)",
     "answer": "48", "sn_ratio": "1:10", "difficulty": "very_hard"},
    {"text": "Chemical formulas: H2O (water), NaCl (salt), CO2 (carbon dioxide), H2SO4 (sulfuric acid), CH4 (methane), C2H5OH (ethanol), NaHCO3 (baking soda), HCl (hydrochloric acid), NH3 (ammonia), CaCO3 (calcium carbonate), Fe2O3 (rust), KMnO4 (potassium permanganate), C6H12O6 (glucose), ATP (adenosine triphosphate), DNA (protein).\nList all the chemical formulas mentioned.",
     "question": "One chemical formula has an incorrect common name. Which substance is incorrectly named? (DNA is not a protein)",
     "answer": "dna", "sn_ratio": "1:10", "difficulty": "very_hard"},
    # Extreme: 1:20 — very subtle anomaly in long list
    {"text": "Prime numbers under 100: 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 51, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97.\nCount the prime numbers listed.",
     "question": "One number in this list is NOT actually prime. Which one? (Hint: 51 = 3 x 17)",
     "answer": "51", "sn_ratio": "1:20", "difficulty": "extreme"},
    {"text": "US State capitals: Alabama-Montgomery, Alaska-Juneau, Arizona-Phoenix, Arkansas-Little Rock, California-Sacramento, Colorado-Denver, Connecticut-Hartford, Delaware-Dover, Florida-Tallahassee, Georgia-Augusta, Hawaii-Honolulu.\nList all state capitals mentioned.",
     "question": "One state capital listed is WRONG. Which state has the incorrect capital? (Hint: Georgia's capital is Atlanta, not Augusta)",
     "answer": "georgia", "sn_ratio": "1:20", "difficulty": "extreme"},
    {"text": "Mathematical constants: pi=3.14159, e=2.71828, sqrt(2)=1.41421, phi=1.61803, sqrt(3)=1.73205, ln(2)=0.69315, sqrt(5)=2.23607, pi/2=1.57080, e^2=7.38906, 1/pi=0.31831, sqrt(7)=2.64575, ln(10)=2.30259, pi^2=9.86960, e*pi=8.53973, phi^2=2.61803, sqrt(10)=3.16228, 2*pi=6.28318, e^pi=23.14069, pi*phi=5.08320.\nVerify these mathematical constants.",
     "question": "One value listed is subtly wrong. Which constant has an incorrect value? (Hint: e*pi should be approximately 8.53974, but check phi^2: it should be 2.61803 = phi+1, that's actually correct. Check e^pi: the actual value is approximately 23.14070.)",
     "answer": "e*pi", "sn_ratio": "1:20", "difficulty": "extreme"},
]

def check_answer(response, answer):
    return answer.lower() in response.strip().lower()

@kbench.task(name="inattentional_blindness",
             description="Tests inattentional blindness via signal-to-noise ratio. "
                         "Primary task distracts while secondary anomaly is embedded. "
                         "Anomaly detection difficulty increases from obvious to extremely subtle. "
                         "Based on Simons & Chabris (1999) gorilla experiment paradigm.")
def inattentional_blindness(llm, text, question, answer, sn_ratio, difficulty) -> float:
    prompt = f"Read the following carefully.\n\n{text}\n\n{question}"
    response = llm.prompt(prompt)
    correct = check_answer(response, answer)
    kbench.assertions.assert_true(correct, expectation=f"Expected '{answer}' at S/N ratio {sn_ratio}")
    return 1.0 if correct else 0.0

df = pd.DataFrame(DATASET)
inattentional_blindness.evaluate(llm=[kbench.llm], evaluation_data=df)


In [ ]:
%choose inattentional_blindness